In [15]:
from pyspark.sql.functions import *

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 17, Finished, Available, Finished, False)

In [16]:
silver_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/silver/orders"
gold_path = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Tables/gold/orders"

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 18, Finished, Available, Finished, False)

In [17]:
# reading from silver
df_silver = spark.readStream.format("delta").load(silver_path)

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 19, Finished, Available, Finished, False)

In [18]:
# aggregation: total sales and total items sold per state per minute
df_gold = (
    df_silver
        .withWatermark("timestamp", "1 minute")
        .groupBy(
            window("timestamp", "1 minute"),
            "state"
        )
        .agg(
            sum("total_amount").alias("total_sales"),
            sum("quantity").alias("total_items")
        )
        .select(
            col("window.start").alias("window_start"),
            col("window.end").alias("window_end"),
            "state",
            "total_sales",
            "total_items"
        )
)

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 20, Finished, Available, Finished, False)

In [19]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 21, Finished, Available, Finished, False)

DataFrame[]

In [20]:
#write to gold table
gold_checkpoint = "abfss://workspace_learn_100046@onelake.dfs.fabric.microsoft.com/lh_ecommerce_orders.Lakehouse/Files/_checkpoints/gold_orders"

query = (
    df_gold.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", gold_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable("gold.orders")
)
query.awaitTermination()

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 22, Finished, Available, Finished, False)

In [21]:
df = spark.read.format("delta").load(gold_path)
print("Gold count:", df.count())
display(df)

StatementMeta(, c5fcccd2-e019-459a-b967-df29b0824997, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90655cd2-5f21-44ff-a49a-1551597c2c37)